# cositos — live widget in an IJulia (Julia Jupyter) kernel

This is the Julia twin of `python_counter.ipynb`. It builds a **cositos** widget in
Julia and displays it over the real Jupyter **comm** channel via IJulia. cositos reuses
anywidget's `AnyModel`/`AnyView` frontend, so the widget renders with **no new
JavaScript** — the exact same counter ESM as the Python notebook.

**Prerequisites** (a live kernel is required — a static `.ipynb` will not render widgets):

- The IJulia kernel must run in a Julia project where **`Cositos`** and **`IJulia`** are
  available. From the repo root:
  `julia -e 'import Pkg; Pkg.activate("/path/to/env"); Pkg.develop(path="julia"); Pkg.add("IJulia")'`
  then launch the kernel bound to that project (`IJulia.installkernel` with
  `env=Dict("JULIA_PROJECT"=>...)`), as `tests/test_e2e_julia.py` does automatically.
- `anywidget` installed in the **frontend** (JupyterLab/Notebook) so the `anywidget`
  model/view module resolves — cositos emits `_model_module="anywidget"`.

In [ ]:
ESM = raw"""
export default {
  render({ model, el }) {
    const btn = document.createElement("button");
    const out = document.createElement("span");
    const paint = () => { out.textContent = " count = " + model.get("count"); };
    btn.textContent = "increment";
    btn.addEventListener("click", () => {
      model.set("count", model.get("count") + 1);
      model.save_changes();
    });
    model.on("change:count", paint);
    paint();
    el.appendChild(btn); el.appendChild(out);
  }
}
"""

In [ ]:
# The host owns the state; cositos is the protocol glue (no observer magic).
using Cositos, IJulia

state = Dict{String,Any}("_esm" => ESM, "count" => 0)
widget = Widget(
    Cositos.ijulia_transport();
    get_state = () -> copy(state),
    set_state = (d) -> merge!(state, d),   # inbound updates from the frontend land here
);   # trailing ; suppresses output — the comm opens when the widget is displayed below

cositos's `CositosIJuliaExt` registers an IJulia display hook, so displaying the
widget renders it and opens its comm automatically — just put it as the last line of a
cell (the Julia analogue of Python's `_repr_mimebundle_`).

In [ ]:
widget   # renders live; opens the comm on first display

Click **increment** in the rendered widget: the frontend calls `save_changes()`, which
sends an `update` back over the comm; `set_state` merges it into `state`. Read it back
from the kernel:

In [ ]:
state["count"]  # reflects clicks made in the frontend